In [2]:
#install dependencies
%pip install --upgrade --quiet \
    "google-genai>=1.51.0" \
    "google-cloud-aiplatform[evaluation]" \
    ipytest pytest \
    "pandas==2.2.2"
# On Colab Enterprise, RESTART THE RUNTIME after the first install (Runtime > Restart
# session), then run from the config cell (Section 1) downward, skipping this cell.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 31.3 MB/s eta 0:00:00


In [3]:
# Configuration
PROJECT_ID      = "qwiklabs-gcp-02-a9a98f0a78c1"

# Gemini (google-genai SDK) settings for the two functions.
GEMINI_LOCATION = "global"
MODEL_ID        = "gemini-2.5-flash"
# Evaluation service location.
EVAL_LOCATION   = "us-central1"

# Enable the API used by the evaluation service.
!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

In [4]:
# INitialize the gemini client
from google import genai
from google.genai import types

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=GEMINI_LOCATION)
print(f"Gen AI SDK client ready - model={MODEL_ID}")

Gen AI SDK client ready - model=gemini-2.5-flash


In [5]:
# First function - classifying a user question
CATEGORIES = ["Employment", "General Information", "Emergency Services", "Tax Related"]
DEFAULT_CATEGORY = "General Information"

CLASSIFIER_PROMPTS = {
    "basic": (
        "Classify the user's question into exactly one of these categories: "
        "Employment, General Information, Emergency Services, Tax Related. "
        "Reply with only the category name."
    ),
    "detailed": (
        "You classify questions for a town government help desk into exactly one category.\n"
        "Categories and their meaning:\n"
        "- Employment: jobs, hiring, applications, payroll, benefits, unemployment.\n"
        "- General Information: hours, locations, events, services, how-to questions.\n"
        "- Emergency Services: police, fire, medical, outages, hazards, urgent safety.\n"
        "- Tax Related: property/income/sales tax, assessments, payments, deadlines.\n"
        "Reply with ONLY the exact category name, nothing else."
    ),
}

def _normalize_category(raw: str) -> str:
    """Map a raw model reply onto one of CATEGORIES (case-insensitive); fall back to default."""
    text = (raw or "").strip().lower()
    for category in CATEGORIES:
        if category.lower() in text:
            return category
    return DEFAULT_CATEGORY

def classify_question(question: str, client=None, model: str = None,
                      prompt_style: str = "detailed") -> str:
    """Classify a question into one of CATEGORIES using Gemini."""
    client = client or genai_client
    model = model or MODEL_ID
    resp = client.models.generate_content(
        model=model,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=CLASSIFIER_PROMPTS[prompt_style],
            temperature=0.0,
            max_output_tokens=16,
        ),
    )
    return _normalize_category(resp.text)

# Quick live check
print(classify_question("How do I apply for a job with the town?"))
print(classify_question("There is a gas leak on Main Street!"))

Employment
General Information


In [18]:
# Second function - generating a social media post
POST_PROMPTS = {
    "basic": "Write a short social media post for this government announcement.",
    "detailed": (
        "You are the communications officer for a town government.\n"
        "Write a clear, calm, public-friendly social media post (under 80 words) for the "
        "announcement below. Include: what is happening, what residents should do, and 2-3 "
        "relevant hashtags. Do not invent specific facts (dates, phone numbers) that are not "
        "given."
    ),
}

def generate_social_post(announcement: str, client=None, model: str = None,
                         prompt_style: str = "detailed") -> str:
    """Generate a social-media post for a government announcement using Gemini."""
    client = client or genai_client
    model = model or MODEL_ID
    resp = client.models.generate_content(
        model=model,
        contents=announcement,
        config=types.GenerateContentConfig(
            system_instruction=POST_PROMPTS[prompt_style],
            temperature=0.7,
            max_output_tokens=1024,
        ),
    )
    return (resp.text or "").strip()

# Quick live check
print(generate_social_post("School is closed tomorrow due to heavy snow and icy roads."))

**Important Update:** Due to heavy snow and icy road conditions, all local schools will be closed tomorrow. Please prioritize safety and avoid unnecessary travel. If you must go out, use extreme caution. Stay warm and safe, everyone!

#SchoolClosure #WinterSafety #CommunityAlert


In [9]:
# import and configure pytest
import ipytest
ipytest.autoconfig()

In [21]:
%%ipytest
# unit tests with pytest
from unittest.mock import MagicMock

def _fake_client(reply_text):
    """A stand-in genai client whose generate_content returns reply_text."""
    client = MagicMock()
    client.models.generate_content.return_value = MagicMock(text=reply_text)
    return client

# ---- _normalize_category (pure function) -----------------------------------
def test_normalize_exact():
    assert _normalize_category("Employment") == "Employment"

def test_normalize_is_case_and_whitespace_insensitive():
    assert _normalize_category("  emergency services\n") == "Emergency Services"

def test_normalize_extracts_from_a_sentence():
    assert _normalize_category("This is Tax Related, I think.") == "Tax Related"

def test_normalize_unknown_falls_back():
    assert _normalize_category("banana") == DEFAULT_CATEGORY
    assert _normalize_category("") == DEFAULT_CATEGORY

# ---- classify_question (mocked model) --------------------------------------
def test_classify_returns_category():
    client = _fake_client("Emergency Services")
    assert classify_question("There's a fire!", client=client) == "Emergency Services"

def test_classify_normalizes_noisy_model_output():
    client = _fake_client("Category: Tax Related.")
    assert classify_question("When are property taxes due?", client=client) == "Tax Related"

def test_classify_passes_question_to_model():
    client = _fake_client("Employment")
    classify_question("How do I apply for a job?", client=client)
    _, kwargs = client.models.generate_content.call_args
    assert kwargs["contents"] == "How do I apply for a job?"

# ---- generate_social_post (mocked model) -----------------------------------
def test_post_returns_stripped_text():
    client = _fake_client("  Snow day! Stay safe.  ")
    out = generate_social_post("School closed", client=client)
    assert out == "Snow day! Stay safe."

def test_post_passes_announcement_to_model():
    client = _fake_client("post text")
    generate_social_post("Holiday on Monday", client=client)
    _, kwargs = client.models.generate_content.call_args
    assert kwargs["contents"] == "Holiday on Monday"

.........                                                                                    [100%]
========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1345
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1345: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
9 passed, 1 warning in 0.03s


In [11]:
#evaluation - comparing prompt designs
import pandas as pd
import vertexai
from vertexai.evaluation import (
    EvalTask,
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
    PairwiseMetric,
)

vertexai.init(project=PROJECT_ID, location=EVAL_LOCATION)
print(f"Evaluation service initialized in {EVAL_LOCATION}")

Evaluation service initialized in us-central1


In [13]:
# Labeled evaluation set (ground truth in `reference`).
classifier_eval = pd.DataFrame([
    {"question": "How do I apply for a job with the city?",            "reference": "Employment"},
    {"question": "What time does the public library open?",            "reference": "General Information"},
    {"question": "There is a car accident with injuries on Highway 1", "reference": "Emergency Services"},
    {"question": "When is my property tax payment due?",               "reference": "Tax Related"},
    {"question": "Where do I file for unemployment benefits?",         "reference": "Employment"},
    {"question": "Is the community pool open on weekends?",            "reference": "General Information"},
    {"question": "My neighbor's house is on fire!",                    "reference": "Emergency Services"},
    {"question": "How do I appeal my home's tax assessment?",          "reference": "Tax Related"},
])

def run_classifier_eval(prompt_style):
    df = classifier_eval.copy()
    df["response"] = [classify_question(q, prompt_style=prompt_style) for q in df["question"]]
    result = EvalTask(dataset=df, metrics=["exact_match"]).evaluate()
    return result

basic_result    = run_classifier_eval("basic")
detailed_result = run_classifier_eval("detailed")

print("Classifier accuracy (exact_match):")
print(f"  basic prompt:    {basic_result.summary_metrics.get('exact_match/mean'):.3f}")
print(f"  detailed prompt: {detailed_result.summary_metrics.get('exact_match/mean'):.3f}")

# Per-row detail for the detailed prompt
detailed_result.metrics_table[["question", "response", "reference", "exact_match/score"]]

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 8 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 8/8 [00:00<00:00, 10.20it/s]
INFO:vertexai.evaluation._evaluation:All 8 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:0.7917930219991831 seconds


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 8 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 8/8 [00:00<00:00, 10.04it/s]
INFO:vertexai.evaluation._evaluation:All 8 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:0.8037374519990408 seconds


Classifier accuracy (exact_match):
  basic prompt:    0.375
  detailed prompt: 0.500


,question,response,reference,exact_match/score
0,How do I apply for a job with the city?,Employment,Employment,1.0
1,What time does the public library open?,General Information,General Information,1.0
2,There is a car accident with injuries on Highw...,General Information,Emergency Services,0.0
3,When is my property tax payment due?,General Information,Tax Related,0.0
4,Where do I file for unemployment benefits?,Employment,Employment,1.0
5,Is the community pool open on weekends?,General Information,General Information,1.0
6,My neighbor's house is on fire!,General Information,Emergency Services,0.0
7,How do I appeal my home's tax assessment?,General Information,Tax Related,0.0


In [19]:
# model based quality metric
import time
from google.genai import errors as genai_errors

post_quality = PointwiseMetric(
    metric="post_quality",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "clarity": "The post is clear, concise, and easy for the public to understand.",
            "actionable": "The post tells residents what they should do or know.",
            "tone": "The tone is calm, professional, and appropriate for a government channel.",
            "hashtags": "The post includes a few relevant, sensible hashtags.",
        },
        rating_rubric={
            "5": "Excellent on all criteria.",
            "3": "Acceptable; meets most criteria with minor gaps.",
            "1": "Poor; misses most criteria.",
        },
    ),
)

# Trimmed to 2 announcements to halve the judge-model calls per eval (lighter on lab quota).
announcements = [
    "A winter storm warning is in effect tonight with 8-12 inches of snow expected.",
    "All public schools are closed tomorrow due to icy road conditions.",
]

def _evaluate_with_retry(eval_task, max_attempts=6):
    """Run EvalTask.evaluate(), backing off on 429 RESOURCE_EXHAUSTED from the judge model."""
    delay = 15
    for attempt in range(1, max_attempts + 1):
        try:
            return eval_task.evaluate()
        except genai_errors.ClientError as e:
            if getattr(e, "code", None) == 429 and attempt < max_attempts:
                print(f"  429 rate limit on judge model - backing off {delay}s (attempt {attempt})")
                time.sleep(delay)
                delay = min(delay * 2, 120)
            else:
                raise

def run_post_eval(prompt_style):
    df = pd.DataFrame({"prompt": announcements})
    df["response"] = [generate_social_post(a, prompt_style=prompt_style) for a in announcements]
    return _evaluate_with_retry(EvalTask(dataset=df, metrics=[post_quality]))

try:
    print("Evaluating 'basic' prompt...")
    basic_posts = run_post_eval("basic")

    print("Pausing 60s before the next eval to stay under the per-minute quota...")
    time.sleep(60)

    print("Evaluating 'detailed' prompt...")
    detailed_posts = run_post_eval("detailed")

    print("\nPost-generator quality (post_quality, 1-5):")
    print(f"  basic prompt:    {basic_posts.summary_metrics.get('post_quality/mean'):.3f}")
    print(f"  detailed prompt: {detailed_posts.summary_metrics.get('post_quality/mean'):.3f}")
    display(detailed_posts.metrics_table[["prompt", "response", "post_quality/score"]])
except Exception as e:
    print("Model-based eval did not complete (often transient judge-model quota):")
    print(f"  {str(e).splitlines()[0]}")


INFO:vertexai.evaluation.metrics.metric_prompt_template:The `input_variables` parameter is empty. Only the `response` column is used for computing this model-based metric.


Evaluating 'basic' prompt...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 2 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 2/2 [00:15<00:00,  7.92s/it]
INFO:vertexai.evaluation._evaluation:All 2 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:15.837597901998379 seconds


Pausing 60s before the next eval to stay under the per-minute quota...
Evaluating 'detailed' prompt...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 2 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 2/2 [00:04<00:00,  2.33s/it]
INFO:vertexai.evaluation._evaluation:All 2 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:4.6733145709986275 seconds



Post-generator quality (post_quality, 1-5):
  basic prompt:    3.000
  detailed prompt: 5.000


,prompt,response,post_quality/score
0,A winter storm warning is in effect tonight wi...,A Winter Storm Warning is in effect for tonigh...,5.0
1,All public schools are closed tomorrow due to ...,**ATTN Residents:** All public schools will be...,5.0


In [20]:
# Head to head prompts - Who will win?
pairwise_quality = PairwiseMetric(
    metric="pairwise_post_quality",
    metric_prompt_template=(
        "You compare two social media posts written for the same government announcement.\n"
        "Pick the better post based on clarity, actionability, appropriate tone, and useful "
        "hashtags.\n\n"
        "Announcement (prompt): {prompt}\n\n"
        "Baseline response: {baseline_model_response}\n\n"
        "Candidate response: {response}\n\n"
        "Which is better?"
    ),
)

try:
    pairwise_df = pd.DataFrame({"prompt": announcements})
    pairwise_df["baseline_model_response"] = [generate_social_post(a, prompt_style="basic")
                                              for a in announcements]
    pairwise_df["response"]                = [generate_social_post(a, prompt_style="detailed")
                                              for a in announcements]
    pairwise_result = EvalTask(dataset=pairwise_df, metrics=[pairwise_quality]).evaluate()
    print("Pairwise win rates (candidate = detailed prompt):")
    for k, v in pairwise_result.summary_metrics.items():
        if "win_rate" in k:
            print(f"  {k}: {v:.3f}")
    display(pairwise_result.metrics_table)
except Exception as e:
    print("Pairwise eval did not complete (often transient judge-model quota):")
    print(f"  {str(e).splitlines()[0]}")

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 2 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 2/2 [00:14<00:00,  7.14s/it]
INFO:vertexai.evaluation._evaluation:All 2 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:14.2911683300008 seconds


Pairwise win rates (candidate = detailed prompt):
  pairwise_post_quality/candidate_model_win_rate: 0.500
  pairwise_post_quality/baseline_model_win_rate: 0.500


,prompt,baseline_model_response,response,pairwise_post_quality/explanation,pairwise_post_quality/pairwise_choice
0,A winter storm warning is in effect tonight wi...,"Here are a few options, pick the one that best...",**Winter Storm Warning in effect tonight.** We...,Post B is better because it provides more comp...,CANDIDATE
1,All public schools are closed tomorrow due to ...,Here are a few options for a social media post...,"Due to icy road conditions, all public schools...",The baseline response (Option 1) is better due...,BASELINE
